<a href="https://colab.research.google.com/github/clee2026/MSDS_498/blob/main/capstone/finding_larger/notebook_5_v3_management_resource_allocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5: Management Resource Allocation and Operational Decision Support

This notebook converts the existing NYC 311 capstone outputs from notebooks 1 to 4 into management-facing metrics, tables, and report-ready visuals. The purpose is not to estimate exact staffing counts. The purpose is to identify operational pressure, service delay risk, repeat complaint burden, hotspot recurrence, and priority areas for proactive resource allocation.

**Main outputs**

- Operational KPI summaries by borough, agency, complaint type, and operational segment
- SLA-style 24-hour and 48-hour resolution metrics
- Operational stress index rankings
- Pareto workload concentration tables and charts
- Hotspot priority rankings
- Trend and seasonal planning outputs
- Management-ready recommendation tables
- A single zip file containing all CSVs, plots, and documentation


In [ ]:
# 0. Setup

from pathlib import Path
import json
import warnings
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Update BASE_DIR if running outside the uploaded file environment.
BASE_DIR = Path("/content/")

INPUTS = {
    "analysis_ready_parquet": BASE_DIR / "analysis_ready_sample_v3.parquet",
    "top_complaint_types": BASE_DIR / "top_values_complaint_type.csv",
    "classification_metrics": BASE_DIR / "classification_metrics_delay_risk_48h.csv",
    "geographic_hotspots": BASE_DIR / "geographic_hotspot_summary.csv",
    "monthly_trends": BASE_DIR / "monthly_trend_by_operational_segment.csv",
    "notebook1_manifest": BASE_DIR / "run_manifest.json",
    "notebook2_manifest": BASE_DIR / "notebook2_v3_manifest.json",
    "notebook3_manifest": BASE_DIR / "notebook3_v3_manifest.json",
    "notebook4_manifest": BASE_DIR / "notebook4_v3_manifest.json",
}

OUTPUT_DIR = BASE_DIR / "notebook5_management_resource_allocation_outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
PLOT_DIR = OUTPUT_DIR / "plots"
DOC_DIR = OUTPUT_DIR / "documentation"

for folder in [TABLE_DIR, PLOT_DIR, DOC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Adjust for chart readability
# Monthly chart readability controls.
# Use 2 to label every other month, 3 to label every third month, etc.
MONTH_LABEL_INTERVAL = 2
MONTH_LABEL_FORMAT = "%Y-%m"

print(f"Output folder: {OUTPUT_DIR}")


Output folder: /content/notebook5_management_resource_allocation_outputs


In [ ]:
# 1. Load inputs and validate availability

missing = [name for name, path in INPUTS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing input files: {missing}")

# pyarrow is required for parquet reading in most environments.
df = pd.read_parquet(INPUTS["analysis_ready_parquet"])
top_complaints = pd.read_csv(INPUTS["top_complaint_types"])
model_metrics = pd.read_csv(INPUTS["classification_metrics"])
hotspots = pd.read_csv(INPUTS["geographic_hotspots"])
trends = pd.read_csv(INPUTS["monthly_trends"])

manifests = {}
for key in ["notebook1_manifest", "notebook2_manifest", "notebook3_manifest", "notebook4_manifest"]:
    with open(INPUTS[key], "r") as f:
        manifests[key] = json.load(f)

print("Analysis-ready sample shape:", df.shape)
print("Hotspot summary shape:", hotspots.shape)
print("Trend summary shape:", trends.shape)
print("Model metrics shape:", model_metrics.shape)


Analysis-ready sample shape: (500000, 52)
Hotspot summary shape: (18325, 7)
Trend summary shape: (152, 6)
Model metrics shape: (3, 9)


In [ ]:
# 2. Prepare operational fields

work = df.copy()

# Standardize important text fields.
for col in ["borough", "agency_name", "complaint_type", "created_ym", "season", "incident_zip", "geo_cell"]:
    if col in work.columns:
        work[col] = work[col].astype("string").fillna("UNKNOWN")

# Build delay and SLA metrics from response_time_hours.
# Open records with missing response times are retained in total workload counts but excluded from closed-response timing averages.
work["has_response_time"] = work["response_time_hours"].notna()
work["delay_risk_24h_flag"] = np.where(work["has_response_time"], (work["response_time_hours"] > 24).astype(int), np.nan)
work["delay_risk_48h_flag"] = np.where(work["has_response_time"], (work["response_time_hours"] > 48).astype(int), np.nan)
work["resolved_within_24h_flag"] = np.where(work["has_response_time"], (work["response_time_hours"] <= 24).astype(int), np.nan)
work["resolved_within_48h_flag"] = np.where(work["has_response_time"], (work["response_time_hours"] <= 48).astype(int), np.nan)

# Cap response time for some plots and stress-score stability.
p99_response = work.loc[work["has_response_time"], "response_time_hours"].quantile(0.99)
work["response_time_hours_capped_p99"] = work["response_time_hours"].clip(upper=p99_response)

# Ensure repeat flag exists.
if "repeat_complaint_proxy" not in work.columns:
    work["repeat_complaint_proxy"] = 0

print("Response time p99 cap:", round(p99_response, 2))
print("Closed-response records:", int(work["has_response_time"].sum()))
print("Total workload records:", len(work))


Response time p99 cap: 12369.49
Closed-response records: 487454
Total workload records: 500000


In [ ]:
# 3. Reusable KPI and stress-score functions

def normalize_series(s):
    s = pd.to_numeric(s, errors="coerce")
    min_v, max_v = s.min(), s.max()
    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(0.0, index=s.index)
    return (s - min_v) / (max_v - min_v)


def build_kpi_table(data, group_cols, min_records=25):
    g = data.groupby(group_cols, dropna=False)
    out = g.agg(
        request_count=("unique_key", "count"),
        closed_response_records=("has_response_time", "sum"),
        avg_response_hours=("response_time_hours", "mean"),
        median_response_hours=("response_time_hours", "median"),
        p90_response_hours=("response_time_hours", lambda s: s.dropna().quantile(0.90) if s.notna().any() else np.nan),
        resolved_within_24h_rate=("resolved_within_24h_flag", "mean"),
        resolved_within_48h_rate=("resolved_within_48h_flag", "mean"),
        delay_24h_rate=("delay_risk_24h_flag", "mean"),
        delay_48h_rate=("delay_risk_48h_flag", "mean"),
        repeat_proxy_rate=("repeat_complaint_proxy", "mean"),
    ).reset_index()
    out = out[out["request_count"] >= min_records].copy()
    out["workload_share"] = out["request_count"] / out["request_count"].sum()

    out["volume_score"] = normalize_series(out["request_count"])
    out["delay_score"] = normalize_series(out["delay_48h_rate"])
    out["response_score"] = normalize_series(out["avg_response_hours"])
    out["repeat_score"] = normalize_series(out["repeat_proxy_rate"])

    out["operational_stress_score"] = (
        0.35 * out["volume_score"] +
        0.30 * out["delay_score"] +
        0.20 * out["response_score"] +
        0.15 * out["repeat_score"]
    )

    out = out.sort_values("operational_stress_score", ascending=False).reset_index(drop=True)
    out["stress_rank"] = np.arange(1, len(out) + 1)

    # Priority tiers based on score distribution.
    q75 = out["operational_stress_score"].quantile(0.75)
    q50 = out["operational_stress_score"].quantile(0.50)
    q25 = out["operational_stress_score"].quantile(0.25)

    def tier(x):
        if x >= q75:
            return "Critical"
        if x >= q50:
            return "High"
        if x >= q25:
            return "Moderate"
        return "Monitor"

    out["management_priority_tier"] = out["operational_stress_score"].apply(tier)
    return out


def save_bar_chart(df_plot, label_col, value_col, title, xlabel, ylabel, path, top_n=15, rotation=45):
    temp = df_plot.head(top_n).copy().iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(temp[label_col].astype(str), temp[value_col])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


def save_line_chart(df_plot, x_col, y_col, group_col, title, ylabel, path):
    """
    Saves a readable monthly line chart.

    Readability controls are set in Section 0:
    - MONTH_LABEL_INTERVAL = 2 labels every other month.
    - Increase to 3 or 6 if the chart is still crowded.
    """
    temp = df_plot.copy()
    temp[x_col] = pd.to_datetime(temp[x_col].astype(str) + "-01", errors="coerce")
    temp = temp.dropna(subset=[x_col]).sort_values(x_col)

    fig, ax = plt.subplots(figsize=(16, 7))

    for name, sub in temp.groupby(group_col):
        sub = sub.sort_values(x_col)
        ax.plot(
            sub[x_col],
            sub[y_col],
            marker="o",
            linewidth=1.8,
            markersize=3.5,
            label=str(name)
        )

    ax.set_title(title)
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)

    # Label every N months instead of every month.
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=MONTH_LABEL_INTERVAL))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(MONTH_LABEL_FORMAT))
    ax.tick_params(axis="x", rotation=45)

    ax.grid(axis="y", alpha=0.3)

    # The legend explains that each curve represents one operational segment.
    ax.legend(
        title=f"Line curves: {group_col}",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=True
    )

    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


In [ ]:
# 4. Operational KPI summaries and stress scores

borough_kpis = build_kpi_table(work, ["borough"], min_records=100)
agency_kpis = build_kpi_table(work, ["agency_name"], min_records=1000)
complaint_kpis = build_kpi_table(work, ["complaint_type"], min_records=1000)
borough_complaint_kpis = build_kpi_table(work, ["borough", "complaint_type"], min_records=500)

borough_kpis.to_csv(TABLE_DIR / "borough_operational_kpi_stress_scores.csv", index=False)
agency_kpis.to_csv(TABLE_DIR / "agency_operational_kpi_stress_scores.csv", index=False)
complaint_kpis.to_csv(TABLE_DIR / "complaint_type_operational_kpi_stress_scores.csv", index=False)
borough_complaint_kpis.to_csv(TABLE_DIR / "borough_complaint_type_priority_matrix.csv", index=False)

print("Top borough stress rankings")
display(borough_kpis.head(10))
print("Top agency stress rankings")
display(agency_kpis.head(10))
print("Top complaint type stress rankings")
display(complaint_kpis.head(10))


Top borough stress rankings


,borough,request_count,closed_response_records,avg_response_hours,median_response_hours,p90_response_hours,resolved_within_24h_rate,resolved_within_48h_rate,delay_24h_rate,delay_48h_rate,repeat_proxy_rate,workload_share,volume_score,delay_score,response_score,repeat_score,operational_stress_score,stress_rank,management_priority_tier
0,BROOKLYN,150017,146580,475.365575,7.965556,470.677833,0.574239,0.655485,0.425761,0.344515,0.486598,0.300034,1.000000,0.635737,0.577497,0.581228,0.743405,1,Critical
1,MANHATTAN,98618,95278,626.026476,7.473750,707.385917,0.579725,0.655524,0.420275,0.344476,0.587783,0.197236,0.599387,0.635269,1.000000,0.884377,0.733023,2,Critical
2,BRONX,108644,106598,269.435400,9.998194,414.086667,0.586127,0.676101,0.413873,0.323899,0.626376,0.217288,0.677532,0.387284,0.000000,1.000000,0.503321,3,High
3,QUEENS,121005,117818,503.405670,4.940556,391.660972,0.634182,0.708236,0.365818,0.291764,0.466187,0.242010,0.773875,0.000000,0.656130,0.520078,0.480094,4,Moderate
4,STATEN ISLAND,21716,21180,551.831606,23.437361,462.326417,0.504344,0.625260,0.495656,0.374740,0.292595,0.043432,0.000000,1.000000,0.791933,0.000000,0.458387,5,Monitor


Top agency stress rankings


,agency_name,request_count,closed_response_records,avg_response_hours,median_response_hours,p90_response_hours,resolved_within_24h_rate,resolved_within_48h_rate,delay_24h_rate,delay_48h_rate,repeat_proxy_rate,workload_share,volume_score,delay_score,response_score,repeat_score,operational_stress_score,stress_rank,management_priority_tier
0,ECONOMIC DEVELOPMENT CORPORATION,4266,3303,7446.775588,1334.932500,26260.783833,0.032698,0.033303,0.967302,0.966697,0.891233,0.008551,0.011882,1.000000,1.000000,1.000000,0.654159,1,Critical
1,DEPARTMENT OF HOUSING PRESERVATION AND DEVELOP...,101898,98920,370.241977,96.675556,888.150194,0.122948,0.286171,0.877052,0.713829,0.599874,0.204238,0.460124,0.737054,0.049180,0.572291,0.477839,2,Critical
2,NEW YORK CITY POLICE DEPARTMENT,219489,219390,4.216737,0.917500,5.271667,0.989571,0.994972,0.010429,0.005028,0.659450,0.439931,1.000000,0.000000,0.000000,0.659747,0.448962,3,Critical
3,DEPARTMENT OF PARKS AND RECREATION,18458,16093,4570.436885,530.642778,15962.143333,0.178338,0.229976,0.821662,0.770024,0.289468,0.036996,0.077039,0.795488,0.613528,0.116620,0.405809,4,Critical
4,TAXI AND LIMOUSINE COMMISSION,4363,3403,1500.929906,1105.731944,3466.353500,0.141346,0.187775,0.858654,0.812225,0.308045,0.008745,0.012327,0.839371,0.201102,0.143891,0.317930,5,High
5,DEPARTMENT OF SANITATION,57192,56585,247.856146,52.922222,243.232667,0.230733,0.457347,0.769267,0.542653,0.288047,0.114632,0.254872,0.559054,0.032736,0.114534,0.280649,6,High
6,DEPARTMENT OF BUILDINGS,14563,14560,1208.074076,215.482361,3423.003028,0.287912,0.349794,0.712088,0.650206,0.248987,0.029189,0.059157,0.670894,0.161753,0.057195,0.262903,7,High
7,DEPARTMENT OF HEALTH AND MENTAL HYGIENE,12172,11099,3772.644425,19.980000,16079.744500,0.517164,0.582395,0.482836,0.417605,0.248521,0.024397,0.048179,0.429022,0.506335,0.056511,0.255313,8,Moderate
8,DEPARTMENT OF TRANSPORTATION,30526,28741,327.187171,42.816667,503.450000,0.417557,0.523573,0.582443,0.476427,0.283791,0.061185,0.132445,0.490189,0.043395,0.108286,0.218334,9,Moderate
9,MAYOR'S OFFICE OF SPECIAL ENFORCEMENT,1678,1648,234.623987,13.677778,768.813556,0.537015,0.598908,0.462985,0.401092,0.434446,0.003363,0.000000,0.411851,0.030958,0.329445,0.179164,10,Moderate


Top complaint type stress rankings


,complaint_type,request_count,closed_response_records,avg_response_hours,median_response_hours,p90_response_hours,resolved_within_24h_rate,resolved_within_48h_rate,delay_24h_rate,delay_48h_rate,repeat_proxy_rate,workload_share,volume_score,delay_score,response_score,repeat_score,operational_stress_score,stress_rank,management_priority_tier
0,NOISE - HELICOPTER,4266,3303,7446.775588,1334.932500,26260.783833,0.032698,0.033303,0.967302,0.966697,0.891233,0.008999,0.051057,0.975025,0.508563,1.000000,0.562090,1,Critical
1,NEW TREE REQUEST,2232,2156,14641.672178,11010.812639,32761.681111,0.006957,0.012059,0.993043,0.987941,0.312276,0.004708,0.019123,0.996452,1.000000,0.305225,0.551412,2,Critical
2,FOOD ESTABLISHMENT,1612,1522,13316.347258,9206.148056,32976.965556,0.006570,0.008541,0.993430,0.991459,0.160050,0.003400,0.009389,1.000000,0.909476,0.122547,0.503563,3,Critical
3,HEAT/HOT WATER,40064,39550,51.530525,43.178333,95.526361,0.255879,0.565284,0.744121,0.434716,0.828050,0.084509,0.613088,0.438461,0.003441,0.924178,0.485434,4,Critical
4,UNSANITARY CONDITION,15611,15033,601.693356,271.138056,1420.390500,0.022284,0.069115,0.977716,0.930885,0.588944,0.032929,0.229174,0.938905,0.041019,0.637239,0.465672,5,Critical
5,ILLEGAL PARKING,64708,64674,2.529265,1.140556,5.706389,0.994882,0.998949,0.005118,0.001051,0.687087,0.136492,1.000000,0.001060,0.000094,0.755015,0.463589,6,Critical
6,NOISE - RESIDENTIAL,58468,58442,8.100879,0.873889,6.272028,0.975138,0.984977,0.024862,0.015023,0.772782,0.123330,0.902032,0.015153,0.000474,0.857853,0.449030,7,Critical
7,MAINTENANCE OR FACILITY,3154,2922,2219.753326,435.833472,6857.087111,0.112252,0.159822,0.887748,0.840178,0.602727,0.006653,0.033598,0.847416,0.151538,0.653779,0.394359,8,Critical
8,PLUMBING,9257,8909,543.895607,204.229722,1373.690778,0.054888,0.144910,0.945112,0.855090,0.509236,0.019526,0.129416,0.862457,0.037071,0.541587,0.392685,9,Critical
9,PAINT/PLASTER,8340,8049,458.372602,188.150000,1056.265333,0.045596,0.138030,0.954404,0.861970,0.512950,0.017592,0.115019,0.869396,0.031230,0.546043,0.389228,10,Critical


In [ ]:
# 5. SLA-style performance summaries

sla_summary = pd.DataFrame({
    "metric": [
        "total_records",
        "closed_response_records",
        "avg_response_hours",
        "median_response_hours",
        "p90_response_hours",
        "resolved_within_24h_rate",
        "resolved_within_48h_rate",
        "delay_24h_rate",
        "delay_48h_rate",
        "repeat_proxy_rate",
    ],
    "value": [
        len(work),
        int(work["has_response_time"].sum()),
        work["response_time_hours"].mean(),
        work["response_time_hours"].median(),
        work["response_time_hours"].quantile(0.90),
        work["resolved_within_24h_flag"].mean(),
        work["resolved_within_48h_flag"].mean(),
        work["delay_risk_24h_flag"].mean(),
        work["delay_risk_48h_flag"].mean(),
        work["repeat_complaint_proxy"].mean(),
    ]
})

sla_by_borough = borough_kpis[[
    "borough", "request_count", "closed_response_records", "avg_response_hours", "median_response_hours",
    "p90_response_hours", "resolved_within_24h_rate", "resolved_within_48h_rate",
    "delay_24h_rate", "delay_48h_rate", "repeat_proxy_rate", "management_priority_tier"
]].copy()

sla_summary.to_csv(TABLE_DIR / "overall_sla_performance_summary.csv", index=False)
sla_by_borough.to_csv(TABLE_DIR / "borough_sla_performance_summary.csv", index=False)

print("Overall SLA performance")
display(sla_summary)


Overall SLA performance


,metric,value
0,total_records,500000.000000
1,closed_response_records,487454.000000
2,avg_response_hours,469.880140
3,median_response_hours,7.616667
4,p90_response_hours,478.211194
5,resolved_within_24h_rate,0.589362
6,resolved_within_48h_rate,0.671438
7,delay_24h_rate,0.410638
8,delay_48h_rate,0.328562
9,repeat_proxy_rate,0.523562


In [ ]:
# 6. Pareto workload concentration analysis

pareto = complaint_kpis.sort_values("request_count", ascending=False).copy()
pareto["cumulative_request_count"] = pareto["request_count"].cumsum()
pareto["cumulative_workload_share"] = pareto["cumulative_request_count"] / pareto["request_count"].sum()
pareto.to_csv(TABLE_DIR / "pareto_complaint_type_workload_concentration.csv", index=False)

# Pareto chart
plot_df = pareto.head(20).copy()
fig, ax1 = plt.subplots(figsize=(14, 7))
bars = ax1.bar(
    plot_df["complaint_type"].astype(str),
    plot_df["request_count"],
    label="Bars: request count by complaint type"
)
ax1.set_ylabel("Request count")
ax1.set_xlabel("Complaint type")
ax1.tick_params(axis="x", rotation=70)

ax2 = ax1.twinx()
line, = ax2.plot(
    plot_df["complaint_type"].astype(str),
    plot_df["cumulative_workload_share"],
    marker="o",
    linewidth=2,
    label="Line: cumulative workload share"
)
ax2.set_ylabel("Cumulative workload share")

# Combined legend explains the bar and line meanings.
ax1.legend([bars, line], ["Bars: request count", "Line: cumulative workload share"], loc="upper left")

plt.title("Pareto Analysis of Complaint Type Workload Concentration")
fig.tight_layout()
plt.savefig(PLOT_DIR / "pareto_complaint_type_workload_concentration.png", dpi=200, bbox_inches="tight")
plt.close()

print("Top complaint type Pareto table")
display(pareto.head(15))


Top complaint type Pareto table


,complaint_type,request_count,closed_response_records,avg_response_hours,median_response_hours,p90_response_hours,resolved_within_24h_rate,resolved_within_48h_rate,delay_24h_rate,delay_48h_rate,repeat_proxy_rate,workload_share,volume_score,delay_score,response_score,repeat_score,operational_stress_score,stress_rank,management_priority_tier,cumulative_request_count,cumulative_workload_share
5,ILLEGAL PARKING,64708,64674,2.529265,1.140556,5.706389,0.994882,0.998949,0.005118,0.001051,0.687087,0.136492,1.000000,0.001060,0.000094,0.755015,0.463589,6,Critical,64708,0.136492
6,NOISE - RESIDENTIAL,58468,58442,8.100879,0.873889,6.272028,0.975138,0.984977,0.024862,0.015023,0.772782,0.123330,0.902032,0.015153,0.000474,0.857853,0.449030,7,Critical,123176,0.259822
3,HEAT/HOT WATER,40064,39550,51.530525,43.178333,95.526361,0.255879,0.565284,0.744121,0.434716,0.828050,0.084509,0.613088,0.438461,0.003441,0.924178,0.485434,4,Critical,163240,0.344331
29,NOISE - STREET/SIDEWALK,25428,25415,3.499302,0.632500,3.158500,0.997088,0.998859,0.002912,0.001141,0.719286,0.053637,0.383301,0.001151,0.000160,0.793656,0.253581,30,High,188668,0.397967
36,BLOCKED DRIVEWAY,24372,24358,3.269520,1.341250,6.347361,0.992035,0.997660,0.007965,0.002340,0.442926,0.051409,0.366722,0.002360,0.000144,0.462012,0.198391,37,Moderate,213040,0.449377
4,UNSANITARY CONDITION,15611,15033,601.693356,271.138056,1420.390500,0.022284,0.069115,0.977716,0.930885,0.588944,0.032929,0.229174,0.938905,0.041019,0.637239,0.465672,5,Critical,228651,0.482306
14,REQUEST LARGE BULKY ITEM COLLECTION,15599,15541,90.072273,63.616667,174.850000,0.004569,0.307767,0.995431,0.692233,0.409770,0.032904,0.228985,0.698197,0.006073,0.422222,0.354152,15,Critical,244250,0.515209
32,STREET CONDITION,11139,10472,100.486642,27.502361,213.722889,0.447765,0.613636,0.552235,0.386364,0.413951,0.023496,0.158963,0.389692,0.006785,0.427240,0.237988,33,High,255389,0.538706
41,WATER SYSTEM,9417,9192,97.378258,4.666667,203.288333,0.703438,0.771758,0.296562,0.228242,0.458639,0.019864,0.131928,0.230208,0.006572,0.480867,0.188682,42,Moderate,264806,0.558569
8,PLUMBING,9257,8909,543.895607,204.229722,1373.690778,0.054888,0.144910,0.945112,0.855090,0.509236,0.019526,0.129416,0.862457,0.037071,0.541587,0.392685,9,Critical,274063,0.578096


In [ ]:
# 7. Hotspot priority analysis

hot = hotspots.copy()
# Keep reportable hotspots only so single-record outliers do not dominate management rankings.
hot = hot[hot["request_count"] >= 100].copy()
for col in ["borough", "complaint_type", "incident_zip"]:
    hot[col] = hot[col].astype("string").fillna("UNKNOWN")

# Normalize and score hotspot priority.
hot["volume_score"] = normalize_series(hot["request_count"])
hot["delay_score"] = normalize_series(hot["delay_48h_rate"])
hot["response_score"] = normalize_series(hot["avg_response_hours"])
hot["repeat_score"] = normalize_series(hot["repeat_proxy_rate"])
hot["hotspot_priority_score"] = (
    0.40 * hot["volume_score"] +
    0.25 * hot["delay_score"] +
    0.20 * hot["repeat_score"] +
    0.15 * hot["response_score"]
)
hot = hot.sort_values("hotspot_priority_score", ascending=False).reset_index(drop=True)
hot["hotspot_rank"] = np.arange(1, len(hot) + 1)
hot["hotspot_label"] = hot["borough"].astype(str) + " | " + hot["incident_zip"].astype(str) + " | " + hot["complaint_type"].astype(str)

hot.to_csv(TABLE_DIR / "hotspot_resource_priority_rankings.csv", index=False)

hot_borough = hot.groupby("borough", as_index=False).agg(
    hotspot_count=("hotspot_rank", "count"),
    total_hotspot_requests=("request_count", "sum"),
    avg_hotspot_delay_48h_rate=("delay_48h_rate", "mean"),
    avg_hotspot_repeat_proxy_rate=("repeat_proxy_rate", "mean"),
    avg_hotspot_priority_score=("hotspot_priority_score", "mean"),
).sort_values("total_hotspot_requests", ascending=False)
hot_borough.to_csv(TABLE_DIR / "borough_hotspot_resource_summary.csv", index=False)

print("Top hotspot priority rankings")
display(hot.head(15))


Top hotspot priority rankings


,borough,incident_zip,complaint_type,request_count,avg_response_hours,repeat_proxy_rate,delay_48h_rate,volume_score,delay_score,response_score,repeat_score,hotspot_priority_score,hotspot_rank,hotspot_label
0,BRONX,10466,NOISE - RESIDENTIAL,9980,30.828429,0.987375,0.082766,1.000000,0.082766,0.001896,0.986161,0.618208,1,BRONX | 10466 | NOISE - RESIDENTIAL
1,QUEENS,11356,NOISE - HELICOPTER,147,16105.648549,1.000000,0.897959,0.004757,0.897959,1.000000,1.000000,0.576393,2,QUEENS | 11356 | NOISE - HELICOPTER
2,MANHATTAN,10023,NOISE - HELICOPTER,994,8306.597850,0.972837,1.000000,0.090486,1.000000,0.515748,0.970225,0.557602,3,MANHATTAN | 10023 | NOISE - HELICOPTER
3,MANHATTAN,10011,NOISE - HELICOPTER,317,13078.147744,0.899054,0.936909,0.021964,0.936909,0.812019,0.889347,0.542685,4,MANHATTAN | 10011 | NOISE - HELICOPTER
4,QUEENS,11414,NOISE - HELICOPTER,507,3824.657057,0.994083,1.000000,0.041194,1.000000,0.237459,0.993514,0.500799,5,QUEENS | 11414 | NOISE - HELICOPTER
5,MANHATTAN,10016,NOISE - HELICOPTER,265,5350.424756,0.958491,0.901887,0.016700,0.901887,0.332196,0.954499,0.472881,6,MANHATTAN | 10016 | NOISE - HELICOPTER
6,QUEENS,11430,TAXI COMPLAINT,135,2266.328761,0.992593,0.992593,0.003543,0.992593,0.140700,0.991880,0.469046,7,QUEENS | 11430 | TAXI COMPLAINT
7,MANHATTAN,10024,NOISE - HELICOPTER,123,10265.155436,0.699187,0.943089,0.002328,0.943089,0.637357,0.670263,0.466360,8,MANHATTAN | 10024 | NOISE - HELICOPTER
8,MANHATTAN,10025,NOISE - HELICOPTER,142,4547.900917,0.852113,0.971831,0.004251,0.971831,0.282366,0.837893,0.454592,9,MANHATTAN | 10025 | NOISE - HELICOPTER
9,BRONX,10452,UNSANITARY CONDITION,546,407.045373,0.813187,0.908425,0.045142,0.908425,0.025255,0.795224,0.407996,10,BRONX | 10452 | UNSANITARY CONDITION


In [ ]:
# 8. Trend and seasonal planning summaries

trend = trends.copy()
trend["created_ym"] = trend["created_ym"].astype(str)
trend["month_date"] = pd.to_datetime(trend["created_ym"] + "-01", errors="coerce")
trend["month"] = trend["month_date"].dt.month

trend_summary = trend.groupby("segment_name", as_index=False).agg(
    months_observed=("created_ym", "nunique"),
    avg_monthly_requests=("request_count", "mean"),
    max_monthly_requests=("request_count", "max"),
    avg_response_hours=("avg_response_hours", "mean"),
    avg_delay_48h_rate=("delay_48h_rate", "mean"),
    avg_repeat_proxy_rate=("repeat_proxy_rate", "mean"),
).sort_values("avg_monthly_requests", ascending=False)
trend_summary.to_csv(TABLE_DIR / "operational_segment_trend_summary.csv", index=False)

monthly_seasonal = trend.groupby("month", as_index=False).agg(
    avg_monthly_requests=("request_count", "mean"),
    avg_response_hours=("avg_response_hours", "mean"),
    avg_delay_48h_rate=("delay_48h_rate", "mean"),
    avg_repeat_proxy_rate=("repeat_proxy_rate", "mean"),
)
monthly_seasonal.to_csv(TABLE_DIR / "seasonal_planning_monthly_summary.csv", index=False)

# Save trend charts.
save_line_chart(trend, "created_ym", "request_count", "segment_name", "Monthly Workload Trend by Operational Segment", "Request count", PLOT_DIR / "monthly_workload_trend_by_operational_segment.png")
save_line_chart(trend, "created_ym", "delay_48h_rate", "segment_name", "Monthly 48-Hour Delay Rate by Operational Segment", "48-hour delay rate", PLOT_DIR / "monthly_delay_rate_trend_by_operational_segment.png")

print("Operational segment trend summary")
display(trend_summary)


Operational segment trend summary


,segment_name,months_observed,avg_monthly_requests,max_monthly_requests,avg_response_hours,avg_delay_48h_rate,avg_repeat_proxy_rate
0,High-volume fast-moving workload,76,6168.026316,8968,264.222568,0.315912,0.534023
1,Long-duration service burden,76,98.697368,239,11387.796678,0.964375,0.397420


In [ ]:
# 9. Report-ready management visualizations

save_bar_chart(borough_kpis, "borough", "operational_stress_score", "Borough Operational Stress Score", "Operational stress score", "Borough", PLOT_DIR / "borough_operational_stress_score.png", top_n=10)
save_bar_chart(agency_kpis, "agency_name", "operational_stress_score", "Top Agency Operational Stress Scores", "Operational stress score", "Agency", PLOT_DIR / "agency_operational_stress_score_top15.png", top_n=15)
save_bar_chart(complaint_kpis, "complaint_type", "operational_stress_score", "Top Complaint Type Operational Stress Scores", "Operational stress score", "Complaint type", PLOT_DIR / "complaint_type_operational_stress_score_top15.png", top_n=15)
save_bar_chart(hot, "hotspot_label", "hotspot_priority_score", "Top Hotspot Resource Priority Scores", "Hotspot priority score", "Hotspot", PLOT_DIR / "hotspot_resource_priority_score_top15.png", top_n=15)

# Workload vs delay scatter for complaint types.
scatter_df = complaint_kpis.head(30).copy()
plt.figure(figsize=(10, 7))
plt.scatter(scatter_df["request_count"], scatter_df["delay_48h_rate"], s=(scatter_df["repeat_proxy_rate"].fillna(0) + 0.05) * 500)
for _, row in scatter_df.head(12).iterrows():
    plt.annotate(str(row["complaint_type"])[:22], (row["request_count"], row["delay_48h_rate"]), fontsize=8)
plt.title("Complaint Workload vs 48-Hour Delay Rate")
plt.xlabel("Request count")
plt.ylabel("48-hour delay rate")
plt.tight_layout()
plt.savefig(PLOT_DIR / "complaint_workload_vs_delay_rate_scatter.png", dpi=200, bbox_inches="tight")
plt.close()

# Borough and complaint type heatmap using pivot table.
heat_source = borough_complaint_kpis.sort_values("request_count", ascending=False).head(50)
pivot = heat_source.pivot_table(index="borough", columns="complaint_type", values="delay_48h_rate", aggfunc="mean")
plt.figure(figsize=(14, 6))
plt.imshow(pivot.fillna(0), aspect="auto")
plt.xticks(range(len(pivot.columns)), pivot.columns.astype(str), rotation=75, ha="right")
plt.yticks(range(len(pivot.index)), pivot.index.astype(str))
plt.colorbar(label="48-hour delay rate")
plt.title("Borough by Complaint Type Delay Risk Heatmap")
plt.tight_layout()
plt.savefig(PLOT_DIR / "borough_complaint_delay_rate_heatmap.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved management visualizations to:", PLOT_DIR)


Saved management visualizations to: /content/notebook5_management_resource_allocation_outputs/plots


In [ ]:
# 10. Management recommendation matrix

priority_matrix = borough_complaint_kpis.head(30).copy()

def recommendation(row):
    if row["management_priority_tier"] == "Critical":
        return "Prioritize proactive staffing, backlog monitoring, and repeat-complaint intervention."
    if row["management_priority_tier"] == "High":
        return "Increase monitoring and prepare targeted operational support during demand spikes."
    if row["management_priority_tier"] == "Moderate":
        return "Monitor performance and review if delay or repeat indicators increase."
    return "Maintain baseline monitoring."

priority_matrix["management_action"] = priority_matrix.apply(recommendation, axis=1)
priority_matrix = priority_matrix[[
    "stress_rank", "borough", "complaint_type", "request_count", "workload_share", "avg_response_hours",
    "delay_48h_rate", "repeat_proxy_rate", "operational_stress_score", "management_priority_tier", "management_action"
]]
priority_matrix.to_csv(TABLE_DIR / "management_resource_priority_recommendations.csv", index=False)

# Model metrics are copied into the output folder for direct report reference.
model_metrics.to_csv(TABLE_DIR / "model_delay_risk_metrics_for_management_reference.csv", index=False)

best_model = model_metrics.sort_values("f1", ascending=False).iloc[0]
model_summary = pd.DataFrame([{
    "best_model_by_f1": best_model["model"],
    "accuracy": best_model["accuracy"],
    "balanced_accuracy": best_model["balanced_accuracy"],
    "precision": best_model["precision"],
    "recall": best_model["recall"],
    "f1": best_model["f1"],
    "roc_auc": best_model["roc_auc"],
    "pr_auc": best_model["pr_auc"],
    "positive_rate_test": best_model["positive_rate_test"],
}])
model_summary.to_csv(TABLE_DIR / "best_delay_risk_model_management_summary.csv", index=False)

print("Management resource priority recommendations")
display(priority_matrix.head(15))
print("Best delay-risk model summary")
display(model_summary)


Management resource priority recommendations


,stress_rank,borough,complaint_type,request_count,workload_share,avg_response_hours,delay_48h_rate,repeat_proxy_rate,operational_stress_score,management_priority_tier,management_action
0,1,MANHATTAN,NOISE - HELICOPTER,2679,0.006085,8541.427744,0.964505,0.892870,0.555568,Critical,"Prioritize proactive staffing, backlog monitor..."
1,2,QUEENS,NEW TREE REQUEST,519,0.001179,18235.360593,0.994024,0.246628,0.534053,Critical,"Prioritize proactive staffing, backlog monitor..."
2,3,QUEENS,NOISE - HELICOPTER,966,0.002194,6613.955964,0.974457,0.960663,0.523356,Critical,"Prioritize proactive staffing, backlog monitor..."
3,4,BROOKLYN,NEW TREE REQUEST,815,0.001851,13942.544065,0.989835,0.339877,0.505207,Critical,"Prioritize proactive staffing, backlog monitor..."
4,5,MANHATTAN,NEW TREE REQUEST,520,0.001181,12817.985449,0.988000,0.300000,0.481511,Critical,"Prioritize proactive staffing, backlog monitor..."
5,6,BRONX,NOISE - RESIDENTIAL,22038,0.050053,15.582719,0.038711,0.884881,0.465015,Critical,"Prioritize proactive staffing, backlog monitor..."
6,7,BROOKLYN,ILLEGAL PARKING,24393,0.055402,2.296012,0.000984,0.715082,0.460376,Critical,"Prioritize proactive staffing, backlog monitor..."
7,8,BRONX,UNSANITARY CONDITION,5232,0.011883,495.473761,0.924846,0.675841,0.457464,Critical,"Prioritize proactive staffing, backlog monitor..."
8,9,MANHATTAN,FOOD ESTABLISHMENT,558,0.001267,12610.923253,0.990584,0.155914,0.457144,Critical,"Prioritize proactive staffing, backlog monitor..."
9,10,BROOKLYN,NOISE - HELICOPTER,567,0.001288,3411.092415,0.960784,0.788360,0.450232,Critical,"Prioritize proactive staffing, backlog monitor..."


Best delay-risk model summary


,best_model_by_f1,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,positive_rate_test
0,logistic_regression_balanced,0.836588,0.861566,0.68396,0.934416,0.789808,0.937395,0.85703,0.328563


In [ ]:
# 11. Output documentation and zip package

inventory_rows = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        inventory_rows.append({
            "relative_path": str(path.relative_to(OUTPUT_DIR)),
            "file_type": path.suffix.replace(".", ""),
            "size_kb": round(path.stat().st_size / 1024, 2),
        })

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(DOC_DIR / "output_inventory.csv", index=False)

readme = f"""# Notebook 5 Management Resource Allocation Outputs

This folder contains management-facing outputs from Notebook 5.

## Purpose

The outputs convert NYC 311 analytical results into concrete operational metrics for leadership decision-making. These outputs support report sections about resource allocation, service delay pressure, hotspot recurrence, repeat complaint burden, and proactive operational planning.

## Key folders

- `tables/`: CSV outputs for report metrics and appendix tables.
- `plots/`: PNG charts suitable for insertion into the report or appendix.
- `documentation/`: Output inventory and run context.

## Most useful report tables

- `tables/borough_operational_kpi_stress_scores.csv`
- `tables/agency_operational_kpi_stress_scores.csv`
- `tables/complaint_type_operational_kpi_stress_scores.csv`
- `tables/borough_complaint_type_priority_matrix.csv`
- `tables/overall_sla_performance_summary.csv`
- `tables/borough_sla_performance_summary.csv`
- `tables/hotspot_resource_priority_rankings.csv`
- `tables/management_resource_priority_recommendations.csv`
- `tables/best_delay_risk_model_management_summary.csv`

## Most useful report plots

- `plots/borough_operational_stress_score.png`
- `plots/complaint_type_operational_stress_score_top15.png`
- `plots/agency_operational_stress_score_top15.png`
- `plots/hotspot_resource_priority_score_top15.png`
- `plots/pareto_complaint_type_workload_concentration.png`
- `plots/complaint_workload_vs_delay_rate_scatter.png`
- `plots/borough_complaint_delay_rate_heatmap.png`
- `plots/monthly_workload_trend_by_operational_segment.png`
- `plots/monthly_delay_rate_trend_by_operational_segment.png`

## Important interpretation note

This notebook does not claim to estimate actual staffing counts or budget requirements. It provides operational prioritization metrics based on workload, delay risk, response time, repeat complaint pressure, hotspot recurrence, and trend behavior.

## Input context

Notebook 1 full dataset rows noted in manifest: {manifests.get('notebook1_manifest', {}).get('num_rows', 'unknown')}
Notebook 2 analysis-ready sample rows noted in manifest: {manifests.get('notebook2_manifest', {}).get('sample_rows', 'unknown')}
Notebook 2 sample date range: {manifests.get('notebook2_manifest', {}).get('sample_min_created_date', 'unknown')} to {manifests.get('notebook2_manifest', {}).get('sample_max_created_date', 'unknown')}
Notebook 4 unique months in sample: {manifests.get('notebook4_manifest', {}).get('unique_months_in_sample', 'unknown')}
"""
(DOC_DIR / "README.md").write_text(readme)

# Save run context.
run_context = {
    "output_dir": str(OUTPUT_DIR),
    "input_files": {k: str(v) for k, v in INPUTS.items()},
    "manifests": manifests,
    "rows_in_analysis_ready_sample": int(len(work)),
    "closed_response_records": int(work["has_response_time"].sum()),
    "response_time_p99_cap": float(p99_response),
}
with open(DOC_DIR / "notebook5_run_context.json", "w") as f:
    json.dump(run_context, f, indent=2, default=str)

# Refresh inventory after documentation files are created.
inventory_rows = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        inventory_rows.append({
            "relative_path": str(path.relative_to(OUTPUT_DIR)),
            "file_type": path.suffix.replace(".", ""),
            "size_kb": round(path.stat().st_size / 1024, 2),
        })
pd.DataFrame(inventory_rows).to_csv(DOC_DIR / "output_inventory.csv", index=False)

zip_base = BASE_DIR / "notebook5_management_resource_allocation_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)

print("Created zip package:", zip_path)
print("Output inventory")
display(pd.read_csv(DOC_DIR / "output_inventory.csv"))


Created zip package: /content/notebook5_management_resource_allocation_outputs.zip
Output inventory


,relative_path,file_type,size_kb
0,documentation/README.md,md,2.15
1,documentation/notebook5_run_context.json,json,3.52
2,documentation/output_inventory.csv,csv,1.38
3,plots/agency_operational_stress_score_top15.png,png,158.71
4,plots/borough_complaint_delay_rate_heatmap.png,png,333.79
5,plots/borough_operational_stress_score.png,png,56.46
6,plots/complaint_type_operational_stress_score_...,png,125.63
7,plots/complaint_workload_vs_delay_rate_scatter...,png,118.00
8,plots/hotspot_resource_priority_score_top15.png,png,181.70
9,plots/monthly_delay_rate_trend_by_operational_...,png,197.62
